In [0]:
%run ../00_common/data_utils

In [0]:
# 统一去除时间字符串末尾时区信息（例如 +08:00 / -05:00 / Z）
def strip_timezone_suffix(col_expr):
    return F.regexp_replace(col_expr.cast(StringType()), r"(?:[+-]\d{2}:\d{2}|Z)$", "")

In [0]:

def calc_consumer_attributes_df(task_id, latest_batch_df):
    # Step 1: 提取 CustomAttribute 数组（字符串）
    df_with_ca = latest_batch_df.select(
        F.col("slndc_id"),
        F.col("task_id"),
        F.col("SLNDC_BATCH_ID"),
        F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.MarketCode").alias("SRAT_MRKT_CODE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.CustomAttributeList.CustomAttribute").alias("ca_array")
    )

    # Step 2: 将 JSON 数组转为 Array[struct]，然后 explode
    # 注意：这里要用 F.from_json + schema，但必须让 schema 支持 @Name/@Value
    schema = StructType([
        StructField("@Name", StringType(), True),
        StructField("@Value", StringType(), True)
    ])

    # 现在解析并展开
    consumer_attributes_df = (
        df_with_ca
        .select(
            F.col("slndc_id"),
            F.col("task_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("SRAT_MRKT_CODE"),
            F.explode(F.from_json("ca_array", ArrayType(schema))).alias("attr")
        )
        .select(
            F.expr("uuid()").alias("SRAT_ID"),
            F.col("slndc_id").alias("SRAT_SRCC_ID"),
            F.col("SRAT_MRKT_CODE"),
            F.col("attr.@Name").alias("SRAT_NAME"),
            F.col("attr.@Value").alias("SRAT_VALUE"),
            F.current_timestamp().alias("SRAT_CREATION_DT"),
            F.lit("ELC").alias("SRAT_CREATIONUID"),
            F.current_timestamp().alias("SRAT_UPDATE_DT"),
            F.lit("ELC").alias("SRAT_UPDATEUID"),
            F.col("SLNDC_BATCH_ID").alias("BATCH_ID"),
            F.col("task_id")
        )
    )
    save_to_target_table(consumer_attributes_df,f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_custom_attributes",f"task_id='{task_id}'")

In [0]:
def calc_skin_concerns_df(task_id, latest_batch_df):
    skin_concerns_df = (
        latest_batch_df
        .select(
            F.col("slndc_id"),
            F.col("task_id"),
            F.col("SLNDC_BATCH_ID"),
            F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.MarketCode").alias("SRSK_MRKT_CODE"),
            F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.SkinConcernsList.SkinConcerns").alias("sc_json")
        )
        # 只处理 sc_json 非 null 且非空数组的情况（explode 会自动跳过 null/[]）
        .select(
            F.expr("uuid()").alias("SRSK_ID"),
            F.col("slndc_id").alias("SRSK_SRCC_ID"),
            F.col("SRSK_MRKT_CODE"),
            F.explode(F.from_json("sc_json", "array<string>")).alias("SRSK_CONCERN_DESC"),
            F.current_timestamp().alias("SRSK_CREATION_DT"),
            F.lit("ELC").alias("SRSK_CREATIONUID"),
            F.current_timestamp().alias("SRSK_UPDATE_DT"),
            F.lit("ELC").alias("SRSK_UPDATEUID"),
            F.col("SLNDC_BATCH_ID").alias("BATCH_ID"),
            F.col("task_id")
        )
    )
    save_to_target_table(skin_concerns_df,f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_skin_concerns",f"task_id='{task_id}'")

In [0]:

def calc_makeup_concerns_df(task_id, latest_batch_df):
    makeup_concerns_df = (
        latest_batch_df
        .select(
            F.col("slndc_id"),
            F.col("task_id"),
            F.col("SLNDC_BATCH_ID"),
            F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.MarketCode").alias("SRMC_MRKT_CODE"),
            F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.MakeUpConcernList.MakeUpConcerns").alias("sc_json")
        )
        # 只处理 sc_json 非 null 且非空数组的情况（explode 会自动跳过 null/[]）
        .select(
            F.expr("uuid()").alias("SRMC_ID"),
            F.col("slndc_id").alias("SRMC_SRCC_ID"),
            F.col("SRMC_MRKT_CODE"),
            F.explode(F.from_json("sc_json", "array<string>")).alias("SRMC_CONCERN_DESC"),
            F.current_timestamp().alias("SRMC_CREATION_DT"),
            F.lit("ELC").alias("SRMC_CREATIONUID"),
            F.current_timestamp().alias("SRMC_UPDATE_DT"),
            F.lit("ELC").alias("SRMC_UPDATEUID"),
            F.col("SLNDC_BATCH_ID").alias("BATCH_ID"),
            F.col("task_id")
        )
    )
    save_to_target_table(makeup_concerns_df,f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_makeup_concerns",f"task_id='{task_id}'")

In [0]:
def calc_hairtype_df(task_id, latest_batch_df):
    hairtype_df = (
        latest_batch_df
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.MarketCode").alias("SRHT_MRKT_CODE"),
            F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.HairTypeList.HairType").alias("sc_json")
        )
        # 只处理 sc_json 非 null 且非空数组的情况（explode 会自动跳过 null/[]）
        .select(
            F.expr("uuid()").alias("SRHT_ID"),
            F.col("slndc_id").alias("SRHT_SRCC_ID"),
            F.col("SRHT_MRKT_CODE"),
            F.explode(F.from_json("sc_json", "array<string>")).alias("SRHT_HAIRTYPE"),
            F.current_timestamp().alias("SRHT_CREATION_DT"),
            F.lit("ELC").alias("SRHT_CREATIONUID"),
            F.current_timestamp().alias("SRHT_UPDATE_DT"),
            F.lit("ELC").alias("SRHT_UPDATEUID"),
            F.col("SLNDC_BATCH_ID").alias("BATCH_ID"),
            F.col("task_id")
        )
    )
    save_to_target_table(hairtype_df,f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_hair_type",f"task_id='{task_id}'")

In [0]:
def calc_hair_concerns_df(task_id, latest_batch_df):
    hair_concerns_df = (
        latest_batch_df
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.MarketCode").alias("SRHC_MRKT_CODE"),
            F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.HairConcernsList.HairConcerns").alias("sc_json")
        )
        # 只处理 sc_json 非 null 且非空数组的情况（explode 会自动跳过 null/[]）
        .select(
            F.expr("uuid()").alias("SRHC_ID"),
            F.col("slndc_id").alias("SRHC_SRCC_ID"),
            F.col("SRHC_MRKT_CODE"),
            F.explode(F.from_json("sc_json", "array<string>")).alias("SRHC_CONCERN_DESC"),
            F.current_timestamp().alias("SRHC_CREATION_DT"),
            F.lit("ELC").alias("SRHC_CREATIONUID"),
            F.current_timestamp().alias("SRHC_UPDATE_DT"),
            F.lit("ELC").alias("SRHC_UPDATEUID"),
            F.col("SLNDC_BATCH_ID").alias("BATCH_ID"),
            F.col("task_id")
        )
    )
    save_to_target_table(hair_concerns_df,f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_hair_concerns",f"task_id='{task_id}'")

In [0]:
def calc_hobby_df(task_id, latest_batch_df):
    # Step 1: 定义Hobby 元素的 schema
    hobby_schema = StructType([
        StructField("HobbyDescription", StringType(), True)
    ])
    
    # Step 2: 解析 Hobby 并展开
    hobby_df = (
        latest_batch_df
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.MarketCode").alias("SRHB_MRKT_CODE"),
            F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.HobbyList.Hobby").alias("hobby_json")
        )
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.col("SRHB_MRKT_CODE"),
            F.explode(F.from_json("hobby_json", ArrayType(hobby_schema))).alias("hobby")
        )
        .select(
            F.expr("uuid()").alias("SRHB_ID"),
            F.col("slndc_id").alias("SRHB_SRCC_ID"),
            F.col("SRHB_MRKT_CODE"),
            F.col("hobby.HobbyDescription").alias("SRHB_HBBY_DESC"),
            F.current_timestamp().alias("SRHB_CREATION_DT"),
            F.lit("ELC").alias("SRHB_CREATIONUID"),
            F.current_timestamp().alias("SRHB_UPDATE_DT"),
            F.lit("ELC").alias("SRHB_UPDATEUID"),
            F.col("SLNDC_BATCH_ID").alias("BATCH_ID"),
            F.col("task_id")
        )
    )
    save_to_target_table(hobby_df,f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_hobby",f"task_id='{task_id}'")

In [0]:
def calc_email_df(task_id, latest_batch_df):
    # Step 1: 定义 source 中 EMedia 元素的 schema
    # 注意：字段名包含特殊字符 "@"，必须用字符串形式引用
    e_media_schema = StructType([
        StructField("@TypeCode", StringType(), True),
        StructField("SourceTimestamp", StringType(), True),
        StructField("Address", StringType(), True),
        StructField("ValidityCode", StringType(), True),
        StructField("Primary", StringType(), True),
        StructField("Appid", StringType(), True),
        StructField("ReferenceEMedia", StructType(
            [
                StructField("@TypeCode", StringType(), True),
                StructField("Address", StringType(), True)
            ]
        ), True)
    ])

    # Step 2: 定义 target 中 EMediaList 元素的 schema
    target_media_schema = StructType([
        StructField("source_Address", StringType(), True),
        StructField("Address", StringType(), True),
        StructField("QUALITY_CODE", StringType(), True),
        StructField("QUALITY_DESC", StringType(), True)
    ])

    # Step 3: 解析 source EMedia 并展开
    source_df = (
        latest_batch_df
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.MarketCode").alias("SRCE_MRKT_CODE"),
            F.get_json_object("slndc_payload", "$.source.Consumer.ContactInformation.EMediaList.EMedia").alias("e_media_raw")
        )
        # 单条："EMedia": { ... }  对象
        # 多条："EMedia": [ { ... }, { ... } ]  数组
        # 输入的单条不是数组，需要把单条变成数组
        .withColumn("e_media_json", F.when(F.col("e_media_raw").startswith("{"), F.concat(F.lit("["), F.col("e_media_raw"), F.lit("]")))
                                           .otherwise(F.col("e_media_raw")))
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.col("SRCE_MRKT_CODE"),
            F.explode(F.from_json("e_media_json", ArrayType(e_media_schema))).alias("e_media")
        )
        .select(
            F.expr("uuid()").alias("SRCE_ID"),
            F.col("slndc_id").alias("SRCE_SRCC_ID"),
            F.col("SRCE_MRKT_CODE"),
            F.col("e_media.@TypeCode").alias("SRCE_EMDT_CODE"),
            strip_timezone_suffix(F.col("e_media.SourceTimestamp")).alias("SRCE_SOURCETIMESTAMP"),
            F.col("e_media.Address").alias("SRCE_ADDRESS_SOURCE"),
            F.col("e_media.ValidityCode").alias("SRCE_VALIDITYCODE"),
            F.col("e_media.Primary").alias("SRCE_PRIMARY_FLAG"),
            F.col("e_media.Appid").alias("SRCE_APPID"),
            F.col("e_media.ReferenceEMedia.@TypeCode").alias("SRCE_REFERENCEEMEDIATYPECODE"),
            F.col("e_media.ReferenceEMedia.Address").alias("SRCE_REFERENCEEMEDIAADDRESS"),
            F.col("SLNDC_BATCH_ID").alias("BATCH_ID"),
            F.col("task_id")
        )
    )

    # Step 4: 解析 target EMedia 并展开
    target_df = (
        latest_batch_df
        .select(
            F.col("slndc_id"),
            F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.MarketCode").alias("SRCE_MRKT_CODE"),
            F.get_json_object("slndc_payload", "$.target.EMediaList").alias("target_json")
        )
        .select(
            "slndc_id",
            F.col("SRCE_MRKT_CODE"),
            F.explode(F.from_json("target_json", ArrayType(target_media_schema))).alias("target_media")
        )
        .select(
            "slndc_id",
            F.col("SRCE_MRKT_CODE"),
            F.col("target_media.source_Address").alias("target_source_address"),
            F.col("target_media.Address").alias("target_address"),
            F.col("target_media.QUALITY_CODE").alias("target_quality_code"),
            F.col("target_media.QUALITY_DESC").alias("target_quality_desc")
        )
    ).dropDuplicates(["slndc_id", "SRCE_MRKT_CODE", "target_source_address"])

    # Step 5: 匹配条件：slndc_id 相同且 SRCE_ADDRESS_SOURCE = target_source_address
    email_df = source_df.join(target_df,
        (source_df["SRCE_SRCC_ID"] == target_df["slndc_id"]) &
        (source_df["SRCE_MRKT_CODE"] == target_df["SRCE_MRKT_CODE"]) &
        (F.coalesce(source_df["SRCE_ADDRESS_SOURCE"], F.lit("UNKNOWN")) == F.coalesce(target_df["target_source_address"], F.lit("UNKNOWN"))),
        "left"
    ).select(
        source_df["SRCE_ID"],
        source_df["SRCE_SRCC_ID"],
        source_df["SRCE_MRKT_CODE"],
        source_df["SRCE_EMDT_CODE"],
        source_df["SRCE_SOURCETIMESTAMP"],
        source_df["SRCE_ADDRESS_SOURCE"],
        source_df["SRCE_VALIDITYCODE"],
        source_df["SRCE_PRIMARY_FLAG"],
        source_df["SRCE_APPID"],
        source_df["SRCE_REFERENCEEMEDIATYPECODE"],
        source_df["SRCE_REFERENCEEMEDIAADDRESS"],
        # 从 target 中取出的三个字段
        F.col("target_address").alias("SRCE_ADDRESS"),
        F.col("target_quality_code").alias("SRCE_QUALITY_CODE"),
        F.col("target_quality_desc").alias("SRCE_QUALITY_DESC"),
        # 固定元数据字段
        F.current_timestamp().alias("SRCE_CREATION_DT"),
        F.lit("ELC").alias("SRCE_CREATIONUID"),
        F.current_timestamp().alias("SRCE_UPDATE_DT"),
        F.lit("ELC").alias("SRCE_UPDATEUID"),
        source_df["BATCH_ID"],
        source_df["task_id"]
    )

    # 额外校验：排除表关联，命中则QUALITY_CODE/QUALITY_DESC置为无效
    exclude_df = spark.table(f"{get_env_config('config_database')}.t_merge_exclude_media_config")
    final_df = email_df.join(
        exclude_df,
        (email_df["SRCE_MRKT_CODE"] == exclude_df["MarketCode"]) & (email_df["SRCE_ADDRESS_SOURCE"] == exclude_df["mediaAddress"]),
        "left"
    ).withColumn(
        "SRCE_QUALITY_CODE",
        F.when(F.col("mediaAddress").isNotNull(), F.lit("inv")).otherwise(F.col("SRCE_QUALITY_CODE"))
    ).withColumn(
        "SRCE_QUALITY_DESC",
        F.when(F.col("mediaAddress").isNotNull(), F.lit("Invalid")).otherwise(F.col("SRCE_QUALITY_DESC"))
    ).drop("MarketCode", "mediaAddress")

    save_to_target_table(final_df,f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_emedia",f"task_id='{task_id}'")

In [0]:
def calc_phone_df(task_id, latest_batch_df):
    # Step 1: 定义 source 中 Phone 元素的 schema
    phone_schema = StructType([
        StructField("@TypeCode", StringType(), True),
        StructField("SourceTimestamp", StringType(), True),
        StructField("PhoneNumber", StringType(), True),
        StructField("ValidityCode", StringType(), True),
        StructField("Primary", StringType(), True)
    ])

    # Step 2: 定义 target 中 Phone 元素的 schema
    target_phone_schema = StructType([
        StructField("source_PhoneNumber", StringType(), True),
        StructField("PhoneNumber", StringType(), True),
        StructField("PHONE_COUNTRY_CODE", StringType(), True),
        StructField("QUALITY_CODE", StringType(), True),
        StructField("QUALITY_DESC", StringType(), True),
        StructField("API_PHONENUMBER", StringType(), True),  # 新增 API_PHONENUMBER 字段
        StructField("API_SOUTHCHINESE_FLAG", BooleanType(), True),  # 新增 API_SOUTHCHINESE_FLAG 字段
        StructField("API_DERIVED_PROVINCE", StringType(), True),  # 新增 API_DERIVED_PROVINCE 字段
        StructField("API_DERIVED_CITY", StringType(), True)  # 新增 API_DERIVED_CITY 字段
    ])

    # Step 3: 解析 source Phone 并展开
    source_df = (
        latest_batch_df
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.MarketCode").alias("SRCP_MRKT_CODE"),
            F.get_json_object("slndc_payload", "$.source.Consumer.ContactInformation.PhoneList.Phone").alias("phone_raw")
        )
        # 输入的单条不是数组，需要把单条变成数组
        .withColumn("phone_json", F.when(F.col("phone_raw").startswith("{"), F.concat(F.lit("["), F.col("phone_raw"), F.lit("]")))
                                           .otherwise(F.col("phone_raw")))
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.col("SRCP_MRKT_CODE"),
            F.explode(F.from_json("phone_json", ArrayType(phone_schema))).alias("phone")
        )
        .select(
            F.expr("uuid()").alias("SRCP_ID"),
            F.col("slndc_id").alias("SRCP_SRCC_ID"),
            F.col("SRCP_MRKT_CODE"),
            F.col("phone.@TypeCode").alias("SRCP_PHNT_CODE"),
            strip_timezone_suffix(F.col("phone.SourceTimestamp")).alias("SRCP_SOURCETIMESTAMP"),
            F.col("phone.PhoneNumber").alias("SRCP_PHONENUMBER_SOURCE"),
            F.col("phone.ValidityCode").alias("SRCP_VALIDITYCODE"),
            F.col("phone.Primary").alias("SRCP_PRIMARY_FLAG"),
            F.col("SLNDC_BATCH_ID").alias("BATCH_ID"),
            F.col("task_id")
        )
    )

    # Step 4: 解析 target Phone 并展开
    target_df = (
        latest_batch_df
        .select(
            F.col("slndc_id"),
            F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.MarketCode").alias("SRCP_MRKT_CODE"),
            F.get_json_object("slndc_payload", "$.target.PhoneList").alias("target_json")
        )
        .select(
            "slndc_id",
            F.col("SRCP_MRKT_CODE"),
            F.explode(F.from_json("target_json", ArrayType(target_phone_schema))).alias("target_phone")
        )
        .select(
            "slndc_id",
            F.col("SRCP_MRKT_CODE"),
            F.col("target_phone.source_PhoneNumber").alias("target_source_phonenumber"),
            F.col("target_phone.PhoneNumber").alias("target_phonenumber"),
            F.col("target_phone.PHONE_COUNTRY_CODE").alias("target_phonecountrycode"),
            F.col("target_phone.QUALITY_CODE").alias("target_quality_code"),
            F.col("target_phone.QUALITY_DESC").alias("target_quality_desc"),
            F.col("target_phone.API_PHONENUMBER").alias("target_api_phonenumber"),  # 新增 API_PHONENUMBER 字段
            F.col("target_phone.API_SOUTHCHINESE_FLAG").alias("target_api_southchinese_flag"),  # 新增 API_SOUTHCHINESE_FLAG 字段
            F.col("target_phone.API_DERIVED_PROVINCE").alias("target_api_derived_province"),  # 新增 API_DERIVED_PROVINCE 字段
            F.col("target_phone.API_DERIVED_CITY").alias("target_api_derived_city")  # 新增 API_DERIVED_CITY 字段
        )
    ).dropDuplicates(["slndc_id", "SRCP_MRKT_CODE", "target_source_phonenumber"])

    # Step 5: 匹配条件：slndc_id 相同且 SRCP_PHONENUMBER_SOURCE = target_source_phonenumber
    phone_df = source_df.join(
        target_df,
        (source_df["SRCP_SRCC_ID"] == target_df["slndc_id"]) &
        (source_df["SRCP_MRKT_CODE"] == target_df["SRCP_MRKT_CODE"]) &
        (F.coalesce(source_df["SRCP_PHONENUMBER_SOURCE"], F.lit("UNKNOWN")) == F.coalesce(target_df["target_source_phonenumber"], F.lit("UNKNOWN"))),
        "left"
    ).select(
        source_df["SRCP_ID"],
        source_df["SRCP_SRCC_ID"],
        source_df["SRCP_MRKT_CODE"],
        source_df["SRCP_PHNT_CODE"],
        source_df["SRCP_SOURCETIMESTAMP"],
        source_df["SRCP_PHONENUMBER_SOURCE"],
        source_df["SRCP_VALIDITYCODE"],
        source_df["SRCP_PRIMARY_FLAG"],
        # 从 target 中取出的字段
        F.col("target_phonenumber").alias("SRCP_PHONENUMBER"),
        F.col("target_phonecountrycode").alias("SRCP_PHONECOUNTRYCODE"),
        F.col("target_quality_code").alias("SRCP_QUALITY_CODE"),
        F.col("target_quality_desc").alias("SRCP_QUALITY_DESC"),
        F.col("target_api_phonenumber").alias("SRCP_API_PHONENUMBER"),  # 新增 API_PHONENUMBER 字段
        F.col("target_api_southchinese_flag").alias("SRCP_API_SOUTHCHINESE_FLAG"),  # 新增 API_SOUTHCHINESE_FLAG 字段
        F.col("target_api_derived_province").alias("SRCP_API_DERIVED_PROVINCE"),  # 新增 API_DERIVED_PROVINCE 字段
        F.col("target_api_derived_city").alias("SRCP_API_DERIVED_CITY"),  # 新增 API_DERIVED_CITY 字段
        # 固定元数据字段
        F.current_timestamp().alias("SRCP_CREATION_DT"),
        F.lit("ELC").alias("SRCP_CREATIONUID"),
        F.current_timestamp().alias("SRCP_UPDATE_DT"),
        F.lit("ELC").alias("SRCP_UPDATEUID"),
        source_df["BATCH_ID"],
        source_df["task_id"]
    )


    # 额外校验：排除表关联，命中则QUALITY_CODE/QUALITY_DESC置为无效
    exclude_df = spark.table(f"{get_env_config('config_database')}.t_merge_exclude_phone_config")
    final_df = phone_df.join(
        exclude_df,
        (phone_df["SRCP_MRKT_CODE"] == exclude_df["MarketCode"]) & (phone_df["SRCP_PHONENUMBER_SOURCE"] == exclude_df["phoneNumber"]),
        "left"
    ).withColumn(
        "SRCP_QUALITY_CODE",
        F.when(F.col("phoneNumber").isNotNull(), F.lit("inv")).otherwise(F.col("SRCP_QUALITY_CODE"))
    ).withColumn(
        "SRCP_QUALITY_DESC",
        F.when(F.col("phoneNumber").isNotNull(), F.lit("Invalid")).otherwise(F.col("SRCP_QUALITY_DESC"))
    ).drop("MarketCode", "phoneNumber")
    
    save_to_target_table(final_df,f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_phone",f"task_id='{task_id}'")

In [0]:
def calc_address_df(task_id, latest_batch_df):
    # Step 1: 定义 source 中 Address 元素的 schema
    address_schema = StructType([
        StructField("@TypeCode", StringType(), True),
        StructField("SourceTimestamp", StringType(), True),
        StructField("Address1", StringType(), True),
        StructField("Address2", StringType(), True),
        StructField("Address3", StringType(), True),
        StructField("CityDescription_local", StringType(), True),
        StructField("ProvinceDescription_local", StringType(), True),
        StructField("CountryCode_ISO3", StringType(), True),
        StructField("PostalCode", StringType(), True),
        StructField("ValidityCode", StringType(), True),
        StructField("Primary", StringType(), True)
    ])

    # Step 2: 定义 target 中 Address 元素的 schema
    target_address_schema = StructType([
        StructField("source_Address1", StringType(), True),
        StructField("QUALITY_CODE", StringType(), True),
        StructField("QUALITY_DESC", StringType(), True)
    ])

    # Step 3: 解析 source Address 并展开
    source_df = (
        latest_batch_df
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.MarketCode").alias("SRCA_MRKT_CODE"),
            F.get_json_object("slndc_payload", "$.source.Consumer.ContactInformation.AddressList.Address").alias("address_raw")
        )
        # 输入的单条不是数组，需要把单条变成数组
        .withColumn("address_json", F.when(F.col("address_raw").startswith("{"), F.concat(F.lit("["), F.col("address_raw"), F.lit("]")))
                                           .otherwise(F.col("address_raw")))
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.col("SRCA_MRKT_CODE"),
            F.explode(F.from_json("address_json", ArrayType(address_schema))).alias("addr")
        )
        .select(
            F.expr("uuid()").alias("SRCA_ID"),
            F.col("slndc_id").alias("SRCA_SRCC_ID"),
            F.col("SRCA_MRKT_CODE"),
            F.col("addr.@TypeCode").alias("SRCA_ADDT_CODE"),
            strip_timezone_suffix(F.col("addr.SourceTimestamp")).alias("SRCA_SOURCETIMESTAMP"),
            F.col("addr.Address1").alias("SRCA_ADDRESS1_SOURCE"),
            F.col("addr.Address2").alias("SRCA_ADDRESS2"),
            F.col("addr.Address3").alias("SRCA_ADDRESS3"),
            F.col("addr.CityDescription_local").alias("SRCA_CITY_LOCALDESC"),
            F.col("addr.ProvinceDescription_local").alias("SRCA_PRVN_LOCALDESC"),
            F.col("addr.CountryCode_ISO3").alias("SRCA_CNTR_ISOALPHA3CODE"),
            F.col("addr.PostalCode").alias("SRCA_POSTALCODE"),
            F.col("addr.ValidityCode").alias("SRCA_VALIDITYCODE"),
            F.col("addr.Primary").alias("SRCA_PRIMARY_FLAG"),
            F.col("SLNDC_BATCH_ID").alias("BATCH_ID"),
            F.col("task_id")
        )
    )

    # Step 4: 解析 target Address 并展开
    target_df = (
        latest_batch_df
        .select(
            F.col("slndc_id"),
            F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.MarketCode").alias("SRCA_MRKT_CODE"),
            F.get_json_object("slndc_payload", "$.target.AddressList").alias("target_json")
        )
        .select(
            "slndc_id",
            F.col("SRCA_MRKT_CODE"),
            F.explode(F.from_json("target_json", ArrayType(target_address_schema))).alias("target_addr")
        )
        .select(
            "slndc_id",
            F.col("SRCA_MRKT_CODE"),
            F.col("target_addr.source_Address1").alias("target_source_address1"),
            F.col("target_addr.QUALITY_CODE").alias("target_quality_code"),
            F.col("target_addr.QUALITY_DESC").alias("target_quality_desc")
        )
    ).dropDuplicates(["slndc_id", "SRCA_MRKT_CODE", "target_source_address1"])

    # Step 5: 匹配条件：slndc_id 相同且 SRCA_ADDRESS1_SOURCE = target_source_address1
    address_df = source_df.join(
        target_df,
        (source_df["SRCA_SRCC_ID"] == target_df["slndc_id"]) &
        (source_df["SRCA_MRKT_CODE"] == target_df["SRCA_MRKT_CODE"]) &
        (F.coalesce(source_df["SRCA_ADDRESS1_SOURCE"], F.lit("UNKNOWN")) == F.coalesce(target_df["target_source_address1"], F.lit("UNKNOWN"))),
        "left"
    ).select(
        source_df["SRCA_ID"],
        source_df["SRCA_SRCC_ID"],
        source_df["SRCA_MRKT_CODE"],
        source_df["SRCA_ADDT_CODE"],
        source_df["SRCA_SOURCETIMESTAMP"],
        source_df["SRCA_ADDRESS1_SOURCE"],
        source_df["SRCA_ADDRESS2"],
        source_df["SRCA_ADDRESS3"],
        source_df["SRCA_CITY_LOCALDESC"],
        source_df["SRCA_PRVN_LOCALDESC"],
        source_df["SRCA_CNTR_ISOALPHA3CODE"],
        source_df["SRCA_POSTALCODE"],
        source_df["SRCA_VALIDITYCODE"],
        source_df["SRCA_PRIMARY_FLAG"],
        # 从 target 中取出的字段
        F.col("target_source_address1").alias("SRCA_ADDRESS1"),
        F.col("target_quality_code").alias("SRCA_QUALITY_CODE"),
        F.col("target_quality_desc").alias("SRCA_QUALITY_DESC"),
        # 固定元数据字段
        F.current_timestamp().alias("SRCA_CREATION_DT"),
        F.lit("ELC").alias("SRCA_CREATIONUID"),
        F.current_timestamp().alias("SRCA_UPDATE_DT"),
        F.lit("ELC").alias("SRCA_UPDATEUID"),
        source_df["BATCH_ID"],
        source_df["task_id"]
    )


    # 额外校验：排除表关联，命中则QUALITY_CODE/QUALITY_DESC置为无效
    exclude_df = spark.table(f"{get_env_config('config_database')}.t_merge_exclude_address_config")
    final_df = address_df.join(
        exclude_df,
        (address_df["SRCA_MRKT_CODE"] == exclude_df["MarketCode"]) & (address_df["SRCA_ADDRESS1_SOURCE"] == exclude_df["Address1"]),
        "left"
    ).withColumn(
        "SRCA_QUALITY_CODE",
        F.when(F.col("Address1").isNotNull(), F.lit("inv")).otherwise(F.col("SRCA_QUALITY_CODE"))
    ).withColumn(
        "SRCA_QUALITY_DESC",
        F.when(F.col("Address1").isNotNull(), F.lit("Invalid")).otherwise(F.col("SRCA_QUALITY_DESC"))
    ).drop("MarketCode", "Address1")
    
    save_to_target_table(final_df,f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_address",f"task_id='{task_id}'")

In [0]:
def calc_optin_df(task_id, latest_batch_df):
    # Step 1: 定义 OptIn 元素的 schema
    optin_schema = StructType([
        StructField("OptInTimestamp", StringType(), True),
        StructField("CommunicationChannelCode", StringType(), True),
        StructField("OptInFlag", StringType(), True)
    ])

    # Step 2: 提取、解析并展开
    optin_df = (
        latest_batch_df
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.MarketCode").alias("SRCO_MRKT_CODE"),
            F.get_json_object("slndc_payload", "$.source.Consumer.OptInList.OptIn").alias("optin_raw")
        )
        # 输入的单条不是数组，需要把单条变成数组
        .withColumn("optin_json", F.when(F.col("optin_raw").startswith("{"), F.concat(F.lit("["), F.col("optin_raw"), F.lit("]")))
                                           .otherwise(F.col("optin_raw")))
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.col("SRCO_MRKT_CODE"),
            F.explode(F.from_json("optin_json", ArrayType(optin_schema))).alias("optin")
        )
        .select(
            F.expr("uuid()").alias("SRCO_ID"),
            F.col("slndc_id").alias("SRCO_SRCC_ID"),
            F.col("SRCO_MRKT_CODE"),
            strip_timezone_suffix(F.col("optin.OptInTimestamp")).alias("SRCO_OPTIN_DT"),
            F.col("optin.CommunicationChannelCode").alias("SRCO_COMM_CODE"),
            F.col("optin.OptInFlag").alias("SRCO_OPTIN_FLAG"),
            F.current_timestamp().alias("SRCO_CREATION_DT"),
            F.lit("ELC").alias("SRCO_CREATIONUID"),
            F.current_timestamp().alias("SRCO_UPDATE_DT"),
            F.lit("ELC").alias("SRCO_UPDATEUID"),
            F.col("SLNDC_BATCH_ID").alias("BATCH_ID"),
            F.col("task_id")
        )
    )
    save_to_target_table(optin_df,f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_optin",f"task_id='{task_id}'")

In [0]:
def calc_cross_brand_df(task_id, latest_batch_df):
    # Step 1: 定义 CrossBrandOptIn 的 struct schema
    cross_brand_optin_schema = StructType([
        StructField("OptInTimestamp", StringType(), True),
        StructField("OptInFlag", StringType(), True)
    ])

    # Step 2: 提取并解析
    cross_brand_df = (
        latest_batch_df
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.MarketCode").alias("SRBO_MRKT_CODE"),
            F.get_json_object("slndc_payload", "$.source.Consumer.CrossBrandOptInList.CrossBrandOptIn").alias("cross_optin_raw")
        )
        # 输入的单条不是数组，需要把单条变成数组
        .withColumn("cross_optin_json", F.when(F.col("cross_optin_raw").startswith("{"), F.concat(F.lit("["), F.col("cross_optin_raw"), F.lit("]")))
                                           .otherwise(F.col("cross_optin_raw")))
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.col("SRBO_MRKT_CODE"),
            F.explode(F.from_json("cross_optin_json", ArrayType(cross_brand_optin_schema))).alias("cross_optin")
        )
        .select(
            F.expr("uuid()").alias("SRBO_ID"),
            F.col("slndc_id").alias("SRBO_SRCC_ID"),
            F.col("SRBO_MRKT_CODE"),
            strip_timezone_suffix(F.col("cross_optin.OptInTimestamp")).alias("SRBO_OPTIN_DT"),
            F.col("cross_optin.OptInFlag").alias("SRBO_OPTIN_FLAG"),
            F.current_timestamp().alias("SRBO_CREATION_DT"),
            F.lit("ELC").alias("SRBO_CREATIONUID"),
            F.current_timestamp().alias("SRBO_UPDATE_DT"),
            F.lit("ELC").alias("SRBO_UPDATEUID"),
            F.col("SLNDC_BATCH_ID").alias("BATCH_ID"),
            F.col("task_id")
        )
    )
    save_to_target_table(cross_brand_df,f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_crossbrand_optin",f"task_id='{task_id}'")

In [0]:
def calc_aux_attr_df(task_id, latest_batch_df):
    # Step 1: 定义 AuxiliaryAttribute 的 struct schema
    aux_attr_schema = StructType([
        StructField("Code", StringType(), True),
        StructField("Description", StringType(), True),
        StructField("MultiValue", StringType(), True),
        StructField("Active", StringType(), True),
        StructField("Value", StringType(), True),
    ])

    # Step 2: 提取、解析并展开
    aux_attr_df = (
        latest_batch_df
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.MarketCode").alias("SRAA_MRKT_CODE"),
            F.get_json_object("slndc_payload", "$.source.Consumer.AuxiliaryAttributeList.AuxiliaryAttribute").alias("aux_attr_raw")
        )
        # 输入的单条不是数组，需要把单条变成数组
        .withColumn("aux_attr_json", F.when(F.col("aux_attr_raw").startswith("{"), F.concat(F.lit("["), F.col("aux_attr_raw"), F.lit("]")))
                                           .otherwise(F.col("aux_attr_raw")))
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.col("SRAA_MRKT_CODE"),
            F.explode(F.from_json("aux_attr_json", ArrayType(aux_attr_schema))).alias("aux")
        )
        .select(
            F.expr("uuid()").alias("SRAA_ID"),
            F.col("slndc_id").alias("SRAA_SRCC_ID"),
            F.col("SRAA_MRKT_CODE"),
            F.col("aux.Code").alias("SRAA_CODE"),
            F.col("aux.Description").alias("SRAA_DESC"),
            F.col("aux.MultiValue").alias("SRAA_MULTIVALUEFLAG"),
            F.col("aux.Value").alias("SRAA_VALUE"),
            F.col("aux.Active").alias("SRAA_ACTIVE_FLAG"),
            F.current_timestamp().alias("SRAA_CREATION_DT"),
            F.lit("ELC").alias("SRAA_CREATIONUID"),
            F.current_timestamp().alias("SRAA_UPDATE_DT"),
            F.lit("ELC").alias("SRAA_UPDATEUID"),
            F.col("SLNDC_BATCH_ID").alias("BATCH_ID"),
            F.col("task_id")
        )
    )
    save_to_target_table(aux_attr_df,f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_auxiliary_attribute",f"task_id='{task_id}'")

In [0]:
def calc_tnc_df(task_id, latest_batch_df):
    # Step 1: 定义 TermsAndCondition 的 struct schema
    tnc_schema = StructType([
        StructField("Code", StringType(), True),
        StructField("Description", StringType(), True),
        StructField("Version", StringType(), True),
        StructField("AcceptedDate", StringType(), True)  # 保持为 string，或后续转 date
    ])

    # Step 2: 提取、解析并展开
    tnc_df = (
        latest_batch_df
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.MarketCode").alias("SRCT_MRKT_CODE"),
            F.get_json_object("slndc_payload", "$.source.Consumer.TermsAndConditionList.TermsAndCondition").alias("tnc_raw")
        )
        # 输入的单条不是数组，需要把单条变成数组
        .withColumn("tnc_json", F.when(F.col("tnc_raw").startswith("{"), F.concat(F.lit("["), F.col("tnc_raw"), F.lit("]")))
                                           .otherwise(F.col("tnc_raw")))
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.col("SRCT_MRKT_CODE"),
            F.explode(F.from_json("tnc_json", ArrayType(tnc_schema))).alias("tnc")
        )
        .select(
            F.expr("uuid()").alias("SRCT_ID"),
            F.col("slndc_id").alias("SRCT_SRCC_ID"),
            F.col("SRCT_MRKT_CODE"),
            F.col("tnc.Code").alias("SRCT_TERMS_CODE"),
            F.col("tnc.Description").alias("SRCT_TERMS_DESCRIPTION"),
            F.col("tnc.Version").alias("SRCT_TERMS_VERSION"),
            F.col("tnc.AcceptedDate").alias("SRCT_TERMS_ACCEPT_DT"),
            F.current_timestamp().alias("SRCT_CREATION_DT"),
            F.lit("ELC").alias("SRCT_CREATIONUID"),
            F.current_timestamp().alias("SRCT_UPDATE_DT"),
            F.lit("ELC").alias("SRCT_UPDATEUID"),
            F.col("SLNDC_BATCH_ID").alias("BATCH_ID"),
            F.col("task_id")
        )
    )
    save_to_target_table(tnc_df,f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_terms",f"task_id='{task_id}'")

In [0]:
def calc_remark_df(task_id, latest_batch_df):
    # Step 1: 定义 Remark 的 struct schema
    remark_schema = StructType([
        StructField("RemarkCode", StringType(), True),
        StructField("RemarksDate", StringType(), True),
        StructField("Remarks", StringType(), True)
    ])

    # Step 2: 提取并解析 Remark 对象
    remark_df = (
        latest_batch_df
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.MarketCode").alias("SRCR_MRKT_CODE"),
            F.get_json_object("slndc_payload", "$.source.Consumer.RemarkList.Remark").alias("remark_raw")
        )
        # 输入的单条不是数组，需要把单条变成数组
        .withColumn("remark_json", F.when(F.col("remark_raw").startswith("{"), F.concat(F.lit("["), F.col("remark_raw"), F.lit("]")))
                                           .otherwise(F.col("remark_raw")))
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.col("SRCR_MRKT_CODE"),
            F.explode(F.from_json("remark_json", ArrayType(remark_schema))).alias("remark")
        )
        .select(
            F.expr("uuid()").alias("SRCR_ID"),
            F.col("slndc_id").alias("SRCR_SRCC_ID"),
            F.col("SRCR_MRKT_CODE"),
            F.col("remark.RemarkCode").alias("SRCR_RMAK_CODE"),
            F.col("remark.RemarksDate").alias("SRCR_RMAK_DT"),
            F.col("remark.Remarks").alias("SRCR_RMAK_DESCRIPTION"),
            F.current_timestamp().alias("SRCR_CREATION_DT"),
            F.lit("ELC").alias("SRCR_CREATIONUID"),
            F.current_timestamp().alias("SRCR_UPDATE_DT"),
            F.lit("ELC").alias("SRCR_UPDATEUID"),
            F.col("SLNDC_BATCH_ID").alias("BATCH_ID"),
            F.col("task_id")
        )
    )
    save_to_target_table(remark_df,f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_remark",f"task_id='{task_id}'")

In [0]:
def calc_customer_group_df(task_id, latest_batch_df):
    # Step 1: 提取 CustomerGroup 数组（作为 JSON 字符串）
    # Step 2: 解析为 Array[String]
    # Step 3: explode 成多行
    customer_group_df = (
        latest_batch_df
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.MarketCode").alias("SRCG_MRKT_CODE"),
            F.get_json_object("slndc_payload", "$.source.Consumer.CustomerGroupList.CustomerGroup").alias("group_json")
        )
        .select(
            F.expr("uuid()").alias("SRCG_ID"),
            F.col("slndc_id").alias("SRCG_SRCC_ID"),
            F.col("SRCG_MRKT_CODE"),
            F.explode(F.from_json("group_json", ArrayType(StringType()))).alias("SRCG_CONSUMER_GRP"),
            F.current_timestamp().alias("SRCG_CREATION_DT"),
            F.lit("ELC").alias("SRCG_CREATIONUID"),
            F.current_timestamp().alias("SRCG_UPDATE_DT"),
            F.lit("ELC").alias("SRCG_UPDATEUID"),
            F.col("SLNDC_BATCH_ID").alias("BATCH_ID"),
            F.col("task_id")
        )
    )
    save_to_target_table(customer_group_df,f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_consumergroup",f"task_id='{task_id}'")

In [0]:
def calc_note_df(task_id, latest_batch_df):
    # Step 1: 定义 Note 的 struct schema
    note_schema = StructType([
        StructField("SeqNum", StringType(), True),
        StructField("Type", StringType(), True),
        StructField("Location", StringType(), True),
        StructField("Note", StringType(), True),
        StructField("CreateDate", StringType(), True),
        StructField("CreateBy", StringType(), True),
        StructField("UpdateDate", StringType(), True),
        StructField("UpdateBy", StringType(), True)
    ])

    # Step 2: 提取、解析并展开
    note_df = (
        latest_batch_df
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.MarketCode").alias("SRNO_MRKT_CODE"),
            F.get_json_object("slndc_payload", "$.source.Consumer.NoteList.Note").alias("note_raw")
        )
        # 输入的单条不是数组，需要把单条变成数组
        .withColumn("note_json", F.when(F.col("note_raw").startswith("{"), F.concat(F.lit("["), F.col("note_raw"), F.lit("]")))
                                           .otherwise(F.col("note_raw")))
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.col("SRNO_MRKT_CODE"),
            F.explode(F.from_json("note_json", ArrayType(note_schema))).alias("note")
        )
        .select(
            F.expr("uuid()").alias("SRNO_ID"),
            F.col("slndc_id").alias("SRNO_SRCC_ID"),
            F.col("SRNO_MRKT_CODE"),
            F.col("note.SeqNum").alias("SRNO_SEQ_NUM"),
            F.col("note.Type").alias("SRNO_TYPE_CODE"),
            F.col("note.Location").alias("SRNO_LOCATION"),
            F.col("note.Note").alias("SRNO_NOTE"),
            F.col("note.CreateDate").alias("SRNO_CREATION_DT"),
            F.col("note.CreateBy").alias("SRNO_CREATIONUID"),
            F.col("note.UpdateDate").alias("SRNO_UPDATE_DT"),
            F.col("note.UpdateBy").alias("SRNO_UPDATEUID"),
            F.col("SLNDC_BATCH_ID").alias("BATCH_ID"),
            F.col("task_id")
        )
    )
    save_to_target_table(note_df,f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_notes",f"task_id='{task_id}'")

In [0]:
def calc_program_df(task_id, latest_batch_df):
    # Step 1: 定义完整的 Program struct schema
    program_schema = StructType([
        StructField("ApplicationTouchPointCode", StringType(), True),
        StructField("ConsumerGroup", StringType(), True),
        StructField("ProgramTypeCode", StringType(), True),
        StructField("ProgramTypeDescription", StringType(), True),
        StructField("ProgramLevelCode", StringType(), True),
        StructField("ProgramLevelDescription", StringType(), True),
        StructField("ProgramSystemIDCode", StringType(), True),
        StructField("ProgramSystemIDDescription", StringType(), True),
        StructField("MembershipNum", StringType(), True),
        StructField("CardNum", StringType(), True),
        StructField("StartTimestamp", StringType(), True),
        StructField("EndTimestamp", StringType(), True),
        StructField("PointsAcquired", StringType(), True),
        StructField("PointsRedeemed", StringType(), True),
        StructField("InitialQuota", StringType(), True),
        StructField("AvailableQuota", StringType(), True),
        StructField("CurrentPointsRedeemed", StringType(), True),
        StructField("NextPointsExpiryTimestamp", StringType(), True),
        StructField("NextPointsExpiry", StringType(), True),
        StructField("CurrentYTDSpending", StringType(), True),
        StructField("P12MSpending", StringType(), True),
        StructField("LifetimeSpending", StringType(), True),
        StructField("SpendingToUpgrade", StringType(), True),
        StructField("SpendingToRenew", StringType(), True),
        StructField("CurrentTierSpending", StringType(), True),
        StructField("TierAchievedTimeStamp", StringType(), True),
        StructField("ProgramNextRenewalTimeStamp", StringType(), True)
    ])

    # Step 2: 提取、解析并展开 Program 数组
    program_df = (
        latest_batch_df
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.MarketCode").alias("SRPG_MRKT_CODE"),
            F.get_json_object("slndc_payload", "$.source.Consumer.ProgramList.Program").alias("program_raw")
        )
        # 输入的单条不是数组，需要把单条变成数组
        .withColumn("program_json", F.when(F.col("program_raw").startswith("{"), F.concat(F.lit("["), F.col("program_raw"), F.lit("]")))
                                           .otherwise(F.col("program_raw")))
        .select(
            F.col("slndc_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("task_id"),
            F.col("SRPG_MRKT_CODE"),
            F.explode(F.from_json("program_json", ArrayType(program_schema))).alias("program")
        )
        .select(
            F.expr("uuid()").alias("SRPG_ID"),
            F.col("slndc_id").alias("SRPG_SRCC_ID"),
            F.col("SRPG_MRKT_CODE"),
            F.col("program.ApplicationTouchPointCode").alias("SRPG_APPLICATION_TOCH_CODE"),
            F.col("program.ConsumerGroup").alias("SRPG_CONSUMER_GRP"),
            F.col("program.ProgramTypeCode").alias("SRPG_PRGT_CODE"),
            F.col("program.ProgramTypeDescription").alias("SRPG_PRGT_DESC"),
            F.col("program.ProgramLevelCode").alias("SRPG_PRGL_CODE"),
            F.col("program.ProgramLevelDescription").alias("SRPG_PRGL_DESC"),
            F.col("program.ProgramSystemIDCode").alias("SRPG_SYSTEM_CODE"),
            F.col("program.ProgramSystemIDDescription").alias("SRPG_SYSTEM_DESC"),
            F.col("program.MembershipNum").alias("SRPG_MEMBERSHIPNUM"),
            F.col("program.CardNum").alias("SRPG_CARDNUM"),
            strip_timezone_suffix(F.col("program.StartTimestamp")).alias("SRPG_START_DT"),
            strip_timezone_suffix(F.col("program.EndTimestamp")).alias("SRPG_END_DT"),
            F.col("program.PointsAcquired").alias("SRPG_ACQUIREDPOINT_NUM"),
            F.col("program.PointsRedeemed").alias("SRPG_REDEEMEDPOINT_NUM"),
            F.col("program.InitialQuota").alias("SRPG_INITIAL_QUOTA"),
            F.col("program.AvailableQuota").alias("SRPG_AVAILABLE_QUOTA"),
            F.col("program.CurrentPointsRedeemed").alias("SRPG_CPREDEEMED_NUM"),
            strip_timezone_suffix(F.col("program.NextPointsExpiryTimestamp")).alias("SRPG_NEXTPOINTSEXPIRY_DT"),
            F.col("program.NextPointsExpiry").alias("SRPG_NEXTPOINTEXPIRY_NUM"),
            F.col("program.CurrentYTDSpending").alias("SRPG_CURRENTYTD"),
            F.col("program.P12MSpending").alias("SRPG_P12SPEND"),
            F.col("program.LifetimeSpending").alias("SRPG_LIFETIME_SPEND"),
            F.col("program.SpendingToUpgrade").alias("SRPG_SPEND_TOUPGRADE"),
            F.col("program.SpendingToRenew").alias("SRPG_SPEND_TORENEW"),
            F.col("program.CurrentTierSpending").alias("SRPG_CURRENTTIER_SPEND"),
            strip_timezone_suffix(F.col("program.TierAchievedTimeStamp")).alias("SRPG_TIERACHIEVED_DT"),
            strip_timezone_suffix(F.col("program.ProgramNextRenewalTimeStamp")).alias("SRPG_PROGRAMNEXTRENEWAL_DT"),
            F.current_timestamp().alias("SRPG_CREATION_DT"),
            F.lit("ELC").alias("SRPG_CREATIONUID"),
            F.current_timestamp().alias("SRPG_UPDATE_DT"),
            F.lit("ELC").alias("SRPG_UPDATEUID"),
            F.col("SLNDC_BATCH_ID").alias("BATCH_ID"),
            F.col("task_id")
        )
    )
    save_to_target_table(program_df,f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_program",f"task_id='{task_id}'")

In [0]:
def get_excluded_df(df, table_type, condition_df, MRKT_CODE, SRCC_ID):
    """
    根据condition表取出需要过滤的数据
    """
    conds = condition_df.filter(F.col("type") == table_type).select("market_code", "condition").collect()

    # 构建排除表达式
    expr = F.lit(False)
    for row in conds:
        market = row.market_code
        cond_str = row.condition
        expr = expr | ((F.col(MRKT_CODE) == market) & F.expr(cond_str))

    excluded_df = df.filter(expr).select(
        F.col(MRKT_CODE).alias("SRCC_MRKT_CODE"),
        F.col(SRCC_ID).alias("SRCC_ID")
    ).distinct()
    return excluded_df

In [0]:
def calc_consumer_df(task_id, latest_batch_df):
    consumer_df = latest_batch_df.select(
        
        # uuid
        F.col("slndc_id").alias("SRCC_ID"),

        # Header
        F.get_json_object("slndc_payload", "$.source.Header.@Action").alias("SRCC_ACTION"),
        strip_timezone_suffix(F.get_json_object("slndc_payload", "$.source.Header.DocumentTimestamp")).alias("SRCC_DOCUMENTTIMESTAMP"),
        F.get_json_object("slndc_payload", "$.source.Header.DocumentUUID").alias("SRCC_DOCUMENTUUID"),

        # Consumer root
        F.get_json_object("slndc_payload", "$.source.Consumer.@RecordUUID").alias("SRCC_RECORDUUID"),

        # SourceSystem
        F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.@Code").alias("SRCC_SRCS_CODE"),
        strip_timezone_suffix(F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.SourceTimestamp")).alias("SRCC_SOURCETIMESTAMP"),
        F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.MarketCode").alias("SRCC_MRKT_CODE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.AffiliateCode").alias("SRCC_AFF_CODE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.DivisionCode").alias("SRCC_DVSN_CODE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.BrandCode").alias("SRCC_BRND_CODE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.ConsumerId").alias("SRCC_CONSUMERID"),

        # PersonalData
        # 清洗后的字段（14个）
        F.get_json_object("slndc_payload", "$.target.Salutation").alias("SRCC_SALUTATION"),
        F.get_json_object("slndc_payload", "$.target.EnglishFirstName").alias("SRCC_ENGLISHFIRSTNAME"),
        F.get_json_object("slndc_payload", "$.target.EnglishMiddleName").alias("SRCC_ENGLISHMIDDLENAME"),
        F.get_json_object("slndc_payload", "$.target.EnglishLastName").alias("SRCC_ENGLISHLASTNAME"),
        F.get_json_object("slndc_payload", "$.target.EnglishFullName").alias("SRCC_ENGLISHFULLNAME"),
        F.get_json_object("slndc_payload", "$.target.LocalFirstName").alias("SRCC_LOCALFIRSTNAME"),
        F.get_json_object("slndc_payload", "$.target.LocalMiddleName").alias("SRCC_LOCALMIDDLENAME"),
        F.get_json_object("slndc_payload", "$.target.LocalLastName").alias("SRCC_LOCALLASTNAME"),
        F.get_json_object("slndc_payload", "$.target.LocalFullName").alias("SRCC_LOCALFULLNAME"),
        F.get_json_object("slndc_payload", "$.target.LocalFirstName2").alias("SRCC_LOCALFIRSTNAME2"),  # 新增 LocalFirstName2 字段
        F.get_json_object("slndc_payload", "$.target.LocalMiddleName2").alias("SRCC_LOCALMIDDLENAME2"),  # 新增 LocalMiddleName2 字段
        F.get_json_object("slndc_payload", "$.target.LocalLastName2").alias("SRCC_LOCALLASTNAME2"),  # 新增 LocalLastName2 字段
        F.get_json_object("slndc_payload", "$.target.LocalFullName2").alias("SRCC_LOCALFULLNAME2"),  # 新增 LocalFullName2 字段
        F.get_json_object("slndc_payload", "$.target.BirthYear").alias("SRCC_BIRTHYEAR"),
        F.get_json_object("slndc_payload", "$.target.WrittenLanguageCode").alias("SRCC_WLNG_CODE"),
        F.get_json_object("slndc_payload", "$.target.englishname_quality_code").alias("SRCC_ENGLISHNAME_QUALITY_CODE"),
        F.get_json_object("slndc_payload", "$.target.localname_quality_code").alias("SRCC_LOCALNAME_QUALITY_CODE"),
        F.get_json_object("slndc_payload", "$.target.localname2_quality_code").alias("SRCC_LOCALNAME2_QUALITY_CODE"),
 
        # PersonalData
        # 清洗前的字段
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.Salutation").alias("SRCC_SALUTATION_SOURCE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.EnglishFirstName").alias("SRCC_ENGLISHFIRSTNAME_SOURCE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.EnglishMiddleName").alias("SRCC_ENGLISHMIDDLENAME_SOURCE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.EnglishLastName").alias("SRCC_ENGLISHLASTNAME_SOURCE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.EnglishFullName").alias("SRCC_ENGLISHFULLNAME_SOURCE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.LocalFirstName").alias("SRCC_LOCALFIRSTNAME_SOURCE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.LocalMiddleName").alias("SRCC_LOCALMIDDLENAME_SOURCE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.LocalLastName").alias("SRCC_LOCALLASTNAME_SOURCE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.LocalFullName").alias("SRCC_LOCALFULLNAME_SOURCE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.LocalFirstName2").alias("SRCC_LOCALFIRSTNAME2_SOURCE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.LocalMiddleName2").alias("SRCC_LOCALMIDDLENAME2_SOURCE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.LocalLastName2").alias("SRCC_LOCALLASTNAME2_SOURCE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.LocalFullName2").alias("SRCC_LOCALFULLNAME2_SOURCE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.GenderCode").alias("SRCC_GNDR_CODE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.BirthDay").alias("SRCC_BIRTHDAY"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.BirthMonth").alias("SRCC_BIRTHMONTH"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.BirthYear").alias("SRCC_BIRTHYEAR_SOURCE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.IdentityNum").alias("SRCC_IDENTITYNUM"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.PassportNum").alias("SRCC_PASSPORTNUM"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.SocialSecurityNum").alias("SRCC_SOCIALSECURITYNUM"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.ConsumerClassCode").alias("SRCC_CLAS_CODE"),
        strip_timezone_suffix(F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.RegDate")).alias("SRCC_REG_DT"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.RegTouchPointCode").alias("SRCC_REGISTRATION_TOCH_CODE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.RegPersonnelCode").alias("SRCC_REGISTRATION_PRSN_CODE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.PreferredTouchPointCode").alias("SRCC_PREFERRED_TOCH_CODE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.AssignedPersonnelCode").alias("SRCC_ASSIGNED_PRSN_CODE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.WrittenLanguageCode").alias("SRCC_WLNG_CODE_SOURCE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.SpokenLanguageCode").alias("SRCC_SLNG_CODE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.ConsumerCountryCode_ISO3").alias("SRCC_CNTR_ISOALPHA3CODE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.EthnicityCode").alias("SRCC_ETHN_CODE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.SkinTypeCode").alias("SRCC_SKNT_CODE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.CivilStatusCode").alias("SRCC_CVLS_CODE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.Company").alias("SRCC_COMPANY"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.Department").alias("SRCC_DEPARTMENT"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.JobTitle").alias("SRCC_JOBTITLE"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.YearlyIncome").alias("SRCC_YEARLYINCOME"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.DoNotContact").alias("SRCC_DONOTCONTACT_FLAG"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.CurrencyCode").alias("SRCC_CURR_CODE"),

        # AgeRange (nested object)
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.AgeRange.AgeFrom").alias("SRCC_AGEFROM"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.AgeRange.AgeTo").alias("SRCC_AGETO"),

        # Other flags
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.ProspectFlag").alias("SRCC_PROSPECT_FLAG"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.PreferredCommChannel").alias("SRCC_PREFERRED_COMM_CHANNEL"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.AnniversaryDate").alias("SRCC_ANNIVERSARY_DT"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.ActiveFlag").alias("SRCC_ACTIVE_FLAG"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.EmailReceiptFlag").alias("SRCC_EMAILRECEIPT_FLAG"),
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.Channel").alias("SRCC_CHANNEL"),
        
        # HairTypeList/HairType → 注意：转为字符串
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.HairTypeList.HairType").alias("SRCC_HAIRT_CODE"),
        
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.Nationality").alias("SRCC_NATIONALITY"),
        strip_timezone_suffix(F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.FirstPurchaseDate")).alias("SRCC_FIRSTPURCHASEDATE"),
        
        # talent没有取，但是表结构有，databricks取值
        F.get_json_object("slndc_payload", "$.source.Consumer.PersonalData.CommercialCustomerFlag").alias("SRCC_COMMERCIAL_FLAG"),

        # 新增line_bind字段
        F.get_json_object("slndc_payload", "$.source.Consumer.BestRecord.UniversalKey").alias("SRCC_UNIVERSALKEY"),
        F.get_json_object("slndc_payload", "$.source.Consumer.BestRecord.MasterConsumerID").alias("SRCC_MASTERCONSUMERID"),
        F.get_json_object("slndc_payload", "$.source.Consumer.BestRecord.SourceSystemCode").alias("SRCC_SOURCESYSTEMCODE"),

        F.current_timestamp().alias("SRCC_HRREQUESTTIMESTAMP"),
        F.lit("Request").alias("SRCC_STATUS"),
        F.current_timestamp().alias("SRCC_CREATION_DT"),
        F.lit("ELC").alias("SRCC_CREATIONUID"),
        F.current_timestamp().alias("SRCC_UPDATE_DT"),
        F.lit("ELC").alias("SRCC_UPDATEUID"),
        F.col("SLNDC_BATCH_ID").alias("BATCH_ID"),
        F.col("batch_number"),
        F.col("task_id"),
        F.get_json_object("slndc_payload", "$.record_id").alias("record_id"),
        F.get_json_object("slndc_payload", "$.target").alias("TARGET_RAW")
    )

    consumer_df.cache()
    print(f"consumer_df count: {consumer_df.count()}")

    """
    1.根据condition表取出需要过滤的数据，将这些数据union起来；
    2.用consumer表关联第1步的df，添加 is_include 字段（能关联上为false，关联不上为true）。
    """
    media_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_emedia").filter(F.col("task_id") == task_id)
    phone_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_phone").filter(F.col("task_id") == task_id)
    address_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_address").filter(F.col("task_id") == task_id)
    condition_df = spark.table(f"{get_env_config('config_database')}.t_clean_condition").filter(F.col("is_active") == True) 
    
    # 添加rakuten_linegift source，rakuten_linegift数据is_include都是true，其他source才需要用condition判断is_include
    # rakuten_linegift_sources = spark.table(f"{get_env_config('config_database')}.t_merge_exclude_consumer_config") \
    #                     .filter(F.col("tmec_type").isin([RAKUTEN, LINEGIFT])).select("tmec_sourcesystemcode").distinct().collect()
    # rakuten_linegift_codes = [row.tmec_sourcesystemcode for row in rakuten_linegift_sources]

    # 分别从四张表获取排除记录，然后union
    excluded_consumer_df = get_excluded_df(consumer_df, CONSUMER, condition_df, "SRCC_MRKT_CODE", "SRCC_ID")
    excluded_media_df = get_excluded_df(media_df, EMAIL, condition_df, "SRCE_MRKT_CODE", "SRCE_SRCC_ID")
    excluded_phone_df = get_excluded_df(phone_df, PHONE, condition_df, "SRCP_MRKT_CODE", "SRCP_SRCC_ID")
    excluded_address_df = get_excluded_df(address_df, ADDRESS, condition_df, "SRCA_MRKT_CODE", "SRCA_SRCC_ID")
    excluded_all_df = excluded_consumer_df.union(excluded_media_df).union(excluded_phone_df).union(excluded_address_df).distinct()

    # 必填字段缺失表达式
    pk_miss_expr = F.lit(False)
    for c in CONSUMER_REQUIRED_FIELD:
        pk_miss_expr = pk_miss_expr | F.col(c).isNull()

    # 关联consumer表，添加is_include
    final_df = (consumer_df.alias("c")
                .join(excluded_all_df.alias("e"), ["SRCC_MRKT_CODE", "SRCC_ID"], "left")
                .withColumn("is_pk_miss", pk_miss_expr)
                .withColumn("is_clear_miss",F.col("TARGET_RAW").isNull())
                .withColumn("is_config_excluded", F.col("e.SRCC_ID").isNotNull())
                .withColumn(
                    "exclude_type",
                    F.when(F.col("is_pk_miss"), F.lit(EXCLUDE_TYPE_BY_PK_MISS))
                    .when(F.col("is_clear_miss"), F.lit(EXCLUDE_TYPE_BY_CLEAR_MISS))
                    .when(F.col("is_config_excluded"), F.lit(EXCLUDE_TYPE_BY_CONFIG))
                    .otherwise(F.lit(None).cast(StringType()))
                )
                .withColumn(
                    "is_include",
                    F.when(F.col("exclude_type").isNull(), F.lit(True)).otherwise(F.lit(False))
                )
                .drop(
                    F.col("e.SRCC_MRKT_CODE"),
                    F.col("e.SRCC_ID"),
                    "TARGET_RAW",
                    "is_pk_miss",
                    "is_clear_miss",
                    "is_config_excluded"
                )
    )

    save_to_target_table(final_df,f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_consumer",f"task_id='{task_id}'")
    consumer_df.unpersist()

In [0]:
def calc_derivedlocation_custom_attributes_df(task_id, latest_batch_df):
    # Step 1: 提取 target.CustomAttributeList 数组（字符串）
    df_with_ca = latest_batch_df.select(
        F.col("slndc_id"),
        F.col("task_id"),
        F.col("SLNDC_BATCH_ID"),
        F.get_json_object("slndc_payload", "$.source.Consumer.SourceSystem.MarketCode").alias("SRAT_MRKT_CODE"),
        F.get_json_object("slndc_payload", "$.target.CustomAttributeList").alias("ca_array")
    )

    # Step 2: 将 JSON 数组转为 Array[struct]，然后 explode
    # 注意：target中的字段是 Name/Value（不带@符号）
    schema = StructType([
        StructField("Name", StringType(), True),
        StructField("Value", StringType(), True)
    ])

    # 现在解析并展开
    derivedlocation_custom_attributes_df = (
        df_with_ca
        .select(
            F.col("slndc_id"),
            F.col("task_id"),
            F.col("SLNDC_BATCH_ID"),
            F.col("SRAT_MRKT_CODE"),
            F.explode(F.from_json("ca_array", ArrayType(schema))).alias("attr")
        )
        .select(
            F.expr("uuid()").alias("SRAT_ID"),
            F.col("slndc_id").alias("SRAT_SRCC_ID"),
            F.col("SRAT_MRKT_CODE"),
            F.col("attr.Name").alias("SRAT_NAME"),
            F.col("attr.Value").alias("SRAT_VALUE"),
            F.current_timestamp().alias("SRAT_CREATION_DT"),
            F.lit("ELC").alias("SRAT_CREATIONUID"),
            F.current_timestamp().alias("SRAT_UPDATE_DT"),
            F.lit("ELC").alias("SRAT_UPDATEUID"),
            F.col("SLNDC_BATCH_ID").alias("BATCH_ID"),
            F.col("task_id")
        )
    )
    if not derivedlocation_custom_attributes_df.isEmpty():
        append_table(derivedlocation_custom_attributes_df,f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_custom_attributes")

def calc_scflag_nonchina_custom_attributes_df(task_id):
    """
    为HKG市场、非中国Derived_Location、没有API验证电话号码的消费者生成South China标志属性
    生成两个属性：'South_China_Phone_Flag' 和 'Derived_South_China_Flag'，值都设为'0'
    """
    # 读取已保存的表
    silver_db = get_env_config('silver_consumer_cleansed_database')
    consumer_df = spark.table(f"{silver_db}.t_clean_consumer").filter(F.col("task_id") == task_id)
    phone_df = spark.table(f"{silver_db}.t_clean_phone").filter(F.col("task_id") == task_id)
    custom_attr_df = spark.table(f"{silver_db}.t_clean_custom_attributes").filter(F.col("task_id") == task_id)
    aux_attr_df = spark.table(f"{silver_db}.t_clean_auxiliary_attribute").filter(F.col("task_id") == task_id)
    
    # 找出有API验证电话号码的消费者ID
    phone_with_api_df = phone_df.filter(F.col("SRCP_API_PHONENUMBER").isNotNull()).select("SRCP_SRCC_ID").distinct()
    
    # 筛选Derived_Location属性
    derived_location_df = custom_attr_df.filter(F.col("SRAT_NAME") == "Derived_Location")
    
    # 筛选South Chinese辅助属性
    south_chinese_df = aux_attr_df.filter(F.col("SRAA_DESC") == "South Chinese")
    
    # 执行JOIN和筛选
    result_df = (
        consumer_df
        # inner join phone
        .join(
            phone_df,
            consumer_df["SRCC_ID"] == phone_df["SRCP_SRCC_ID"],
            "inner"
        )
        # inner join custom_attributes (Derived_Location)
        .join(
            derived_location_df,
            consumer_df["SRCC_ID"] == derived_location_df["SRAT_SRCC_ID"],
            "inner"
        )
        # left join auxiliary_attribute (South Chinese)
        .join(
            south_chinese_df,
            consumer_df["SRCC_ID"] == south_chinese_df["SRAA_SRCC_ID"],
            "left"
        )
        # 筛选条件
        .filter(F.col("SRCC_MRKT_CODE") == "HKG")
        .filter(F.coalesce(F.col("SRCP_PHONENUMBER"), F.lit("")) != "")
        .filter(F.col("SRAT_VALUE") != "CHN")
        .filter(
            (F.coalesce(F.col("SRAA_VALUE"), F.lit("")) == "") |
            (~F.col("SRAA_VALUE").isin(["Y", "N"]))
        )
        # 反连接：排除有API电话号码的消费者
        .join(
            phone_with_api_df,
            consumer_df["SRCC_ID"] == phone_with_api_df["SRCP_SRCC_ID"],
            "left_anti"
        )
        .select(
            F.col("SRCC_ID"),
            F.col("SRCC_MRKT_CODE"),
            consumer_df["BATCH_ID"],
            consumer_df["task_id"]
        )
        .distinct()
    )
    
    # 生成两个属性记录：South_China_Phone_Flag 和 Derived_South_China_Flag
    flag1_df = result_df.select(
        F.expr("uuid()").alias("SRAT_ID"),
        F.col("SRCC_ID").alias("SRAT_SRCC_ID"),
        F.col("SRCC_MRKT_CODE").alias("SRAT_MRKT_CODE"),
        F.lit("South_China_Phone_Flag").alias("SRAT_NAME"),
        F.lit('0').alias("SRAT_VALUE"),
        F.current_timestamp().alias("SRAT_CREATION_DT"),
        F.lit("ELC").alias("SRAT_CREATIONUID"),
        F.current_timestamp().alias("SRAT_UPDATE_DT"),
        F.lit("ELC").alias("SRAT_UPDATEUID"),
        F.col("BATCH_ID"),
        F.col("task_id")
    )
    
    flag2_df = result_df.select(
        F.expr("uuid()").alias("SRAT_ID"),
        F.col("SRCC_ID").alias("SRAT_SRCC_ID"),
        F.col("SRCC_MRKT_CODE").alias("SRAT_MRKT_CODE"),
        F.lit("Derived_South_China_Flag").alias("SRAT_NAME"),
        F.lit('0').alias("SRAT_VALUE"),
        F.current_timestamp().alias("SRAT_CREATION_DT"),
        F.lit("ELC").alias("SRAT_CREATIONUID"),
        F.current_timestamp().alias("SRAT_UPDATE_DT"),
        F.lit("ELC").alias("SRAT_UPDATEUID"),
        F.col("BATCH_ID"),
        F.col("task_id")
    )
    
    # 合并两个属性
    scflag_nonchina_custom_attributes_df = flag1_df.unionByName(flag2_df)

    if not scflag_nonchina_custom_attributes_df.isEmpty():
        append_table(scflag_nonchina_custom_attributes_df,f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_custom_attributes")

def calc_scflag_china_custom_attributes_df(task_id):
    """
    为HKG市场、有API验证电话号码的消费者生成API返回的South China相关属性
    生成4个属性：
    1. South_China_Phone_Flag = API返回的southchinese_flag
    2. Derived_Province = API返回的省份
    3. Derived_City = API返回的城市
    4. Derived_South_China_Flag = 基于API标志和Derived_Location计算
    """
    # 读取已保存的表
    silver_db = get_env_config('silver_consumer_cleansed_database')
    consumer_df = spark.table(f"{silver_db}.t_clean_consumer").filter(F.col("task_id") == task_id)
    phone_df = spark.table(f"{silver_db}.t_clean_phone").filter(F.col("task_id") == task_id)
    custom_attr_df = spark.table(f"{silver_db}.t_clean_custom_attributes").filter(F.col("task_id") == task_id)
    
    # Inner join consumer + phone，筛选HKG市场且有API电话号码的记录
    base_df = (
        consumer_df
        .join(
            phone_df,
            consumer_df["SRCC_ID"] == phone_df["SRCP_SRCC_ID"],
            "inner"
        )
        .filter(F.col("SRCC_MRKT_CODE") == "HKG")
        .filter(F.col("SRCP_API_PHONENUMBER").isNotNull())
        .select(
            consumer_df["SRCC_ID"],
            consumer_df["SRCC_MRKT_CODE"],
            phone_df["SRCP_API_SOUTHCHINESE_FLAG"],
            phone_df["SRCP_API_DERIVED_PROVINCE"],
            phone_df["SRCP_API_DERIVED_CITY"],
            consumer_df["BATCH_ID"],
            consumer_df["task_id"]
        )
        .distinct()
    )
    
    # 属性1: South_China_Phone_Flag = API返回的southchinese_flag (转为字符串)
    attr1_df = base_df.select(
        F.expr("uuid()").alias("SRAT_ID"),
        F.col("SRCC_ID").alias("SRAT_SRCC_ID"),
        F.col("SRCC_MRKT_CODE").alias("SRAT_MRKT_CODE"),
        F.lit("South_China_Phone_Flag").alias("SRAT_NAME"),
        F.when(F.col("SRCP_API_SOUTHCHINESE_FLAG") == True, "1")
         .when(F.col("SRCP_API_SOUTHCHINESE_FLAG") == False, "0")
         .otherwise("").alias("SRAT_VALUE"),
        F.current_timestamp().alias("SRAT_CREATION_DT"),
        F.lit("ELC").alias("SRAT_CREATIONUID"),
        F.current_timestamp().alias("SRAT_UPDATE_DT"),
        F.lit("ELC").alias("SRAT_UPDATEUID"),
        F.col("BATCH_ID"),
        F.col("task_id")
    )
    
    # 属性2: Derived_Province = API返回的省份
    attr2_df = base_df.select(
        F.expr("uuid()").alias("SRAT_ID"),
        F.col("SRCC_ID").alias("SRAT_SRCC_ID"),
        F.col("SRCC_MRKT_CODE").alias("SRAT_MRKT_CODE"),
        F.lit("Derived_Province").alias("SRAT_NAME"),
        F.coalesce(F.col("SRCP_API_DERIVED_PROVINCE"), F.lit("")).alias("SRAT_VALUE"),
        F.current_timestamp().alias("SRAT_CREATION_DT"),
        F.lit("ELC").alias("SRAT_CREATIONUID"),
        F.current_timestamp().alias("SRAT_UPDATE_DT"),
        F.lit("ELC").alias("SRAT_UPDATEUID"),
        F.col("BATCH_ID"),
        F.col("task_id")
    )
    
    # 属性3: Derived_City = API返回的城市
    attr3_df = base_df.select(
        F.expr("uuid()").alias("SRAT_ID"),
        F.col("SRCC_ID").alias("SRAT_SRCC_ID"),
        F.col("SRCC_MRKT_CODE").alias("SRAT_MRKT_CODE"),
        F.lit("Derived_City").alias("SRAT_NAME"),
        F.coalesce(F.col("SRCP_API_DERIVED_CITY"), F.lit("")).alias("SRAT_VALUE"),
        F.current_timestamp().alias("SRAT_CREATION_DT"),
        F.lit("ELC").alias("SRAT_CREATIONUID"),
        F.current_timestamp().alias("SRAT_UPDATE_DT"),
        F.lit("ELC").alias("SRAT_UPDATEUID"),
        F.col("BATCH_ID"),
        F.col("task_id")
    )
    
    # 属性4: Derived_South_China_Flag = 基于API标志和Derived_Location计算
    # 需要join custom_attributes获取Derived_Location
    derived_location_df = custom_attr_df.filter(F.col("SRAT_NAME") == "Derived_Location")
    
    attr4_base_df = (
        base_df
        .join(
            derived_location_df.select("SRAT_SRCC_ID", F.col("SRAT_VALUE").alias("derived_location_value")),
            base_df["SRCC_ID"] == derived_location_df["SRAT_SRCC_ID"],
            "inner"
        )
    )
    
    # 如果 API显示是南方人(=1/true) 且 Derived_Location 为 'CHN' 或空，则为 '1'，否则为 '0'
    attr4_df = attr4_base_df.select(
        F.expr("uuid()").alias("SRAT_ID"),
        F.col("SRCC_ID").alias("SRAT_SRCC_ID"),
        F.col("SRCC_MRKT_CODE").alias("SRAT_MRKT_CODE"),
        F.lit("Derived_South_China_Flag").alias("SRAT_NAME"),
        F.when(
            (F.col("SRCP_API_SOUTHCHINESE_FLAG") == True) & 
            (F.coalesce(F.col("derived_location_value"), F.lit("")).isin(["CHN", ""])),
            "1"
        ).otherwise("0").alias("SRAT_VALUE"),
        F.current_timestamp().alias("SRAT_CREATION_DT"),
        F.lit("ELC").alias("SRAT_CREATIONUID"),
        F.current_timestamp().alias("SRAT_UPDATE_DT"),
        F.lit("ELC").alias("SRAT_UPDATEUID"),
        F.col("BATCH_ID"),
        F.col("task_id")
    )
    
    # 合并所有4个属性
    scflag_china_custom_attributes_df = attr1_df.unionByName(attr2_df).unionByName(attr3_df).unionByName(attr4_df)

    if not scflag_china_custom_attributes_df.isEmpty():
        append_table(scflag_china_custom_attributes_df,f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_custom_attributes")

def calc_scflag_auxiliary_attributes_df(task_id):
    """
    为HKG市场、有South Chinese辅助属性、但尚未生成对应自定义属性的消费者补充生成标志
    生成2个属性：
    1. South_China_Phone_Flag = 基于South Chinese辅助属性值
    2. Derived_South_China_Flag = 基于South Chinese辅助属性和Derived_Location
    """
    # 读取已保存的表
    silver_db = get_env_config('silver_consumer_cleansed_database')
    consumer_df = spark.table(f"{silver_db}.t_clean_consumer").filter(F.col("task_id") == task_id)
    aux_attr_df = spark.table(f"{silver_db}.t_clean_auxiliary_attribute").filter(F.col("task_id") == task_id)
    custom_attr_df = spark.table(f"{silver_db}.t_clean_custom_attributes").filter(F.col("task_id") == task_id)
    
    # 筛选South Chinese辅助属性
    south_chinese_df = aux_attr_df.filter(F.col("SRAA_DESC") == "South Chinese")
    
    # 找出已有South_China_Phone_Flag的消费者ID
    existing_sc_phone_flag_df = custom_attr_df.filter(
        F.col("SRAT_NAME") == "South_China_Phone_Flag"
    ).select("SRAT_SRCC_ID").distinct()
    
    # 找出已有Derived_South_China_Flag的消费者ID
    existing_derived_sc_flag_df = custom_attr_df.filter(
        F.col("SRAT_NAME") == "Derived_South_China_Flag"
    ).select("SRAT_SRCC_ID").distinct()
    
    # 属性1: South_China_Phone_Flag
    # 筛选条件：HKG市场 + 有South Chinese辅助属性 + 还没有South_China_Phone_Flag
    attr1_base_df = (
        consumer_df
        .join(
            south_chinese_df,
            consumer_df["SRCC_ID"] == south_chinese_df["SRAA_SRCC_ID"],
            "inner"
        )
        .filter(F.col("SRCC_MRKT_CODE") == "HKG")
        # 排除已有South_China_Phone_Flag的消费者
        .join(
            existing_sc_phone_flag_df,
            consumer_df["SRCC_ID"] == existing_sc_phone_flag_df["SRAT_SRCC_ID"],
            "left_anti"
        )
        .select(
            consumer_df["SRCC_ID"],
            consumer_df["SRCC_MRKT_CODE"],
            south_chinese_df["SRAA_VALUE"],
            consumer_df["BATCH_ID"],
            consumer_df["task_id"]
        )
        .distinct()
    )
    
    attr1_df = attr1_base_df.select(
        F.expr("uuid()").alias("SRAT_ID"),
        F.col("SRCC_ID").alias("SRAT_SRCC_ID"),
        F.col("SRCC_MRKT_CODE").alias("SRAT_MRKT_CODE"),
        F.lit("South_China_Phone_Flag").alias("SRAT_NAME"),
        F.when(F.col("SRAA_VALUE") == "Y", "1").otherwise("0").alias("SRAT_VALUE"),
        F.current_timestamp().alias("SRAT_CREATION_DT"),
        F.lit("ELC").alias("SRAT_CREATIONUID"),
        F.current_timestamp().alias("SRAT_UPDATE_DT"),
        F.lit("ELC").alias("SRAT_UPDATEUID"),
        F.col("BATCH_ID"),
        F.col("task_id")
    )
    
    # 属性2: Derived_South_China_Flag
    # 筛选条件：HKG市场 + 有South Chinese辅助属性 + 有Derived_Location + 还没有Derived_South_China_Flag
    derived_location_df = custom_attr_df.filter(F.col("SRAT_NAME") == "Derived_Location")
    
    attr2_base_df = (
        consumer_df
        .join(
            south_chinese_df,
            consumer_df["SRCC_ID"] == south_chinese_df["SRAA_SRCC_ID"],
            "inner"
        )
        .join(
            derived_location_df.select("SRAT_SRCC_ID", F.col("SRAT_VALUE").alias("derived_location_value")),
            consumer_df["SRCC_ID"] == derived_location_df["SRAT_SRCC_ID"],
            "inner"
        )
        .filter(F.col("SRCC_MRKT_CODE") == "HKG")
        # 排除已有Derived_South_China_Flag的消费者
        .join(
            existing_derived_sc_flag_df,
            consumer_df["SRCC_ID"] == existing_derived_sc_flag_df["SRAT_SRCC_ID"],
            "left_anti"
        )
        .select(
            consumer_df["SRCC_ID"],
            consumer_df["SRCC_MRKT_CODE"],
            south_chinese_df["SRAA_VALUE"],
            F.col("derived_location_value"),
            consumer_df["BATCH_ID"],
            consumer_df["task_id"]
        )
        .distinct()
    )
    
    # 如果 South Chinese = 'Y' 且 Derived_Location 为 'CHN' 或空，则为 '1'，否则为 '0'
    attr2_df = attr2_base_df.select(
        F.expr("uuid()").alias("SRAT_ID"),
        F.col("SRCC_ID").alias("SRAT_SRCC_ID"),
        F.col("SRCC_MRKT_CODE").alias("SRAT_MRKT_CODE"),
        F.lit("Derived_South_China_Flag").alias("SRAT_NAME"),
        F.when(
            (F.col("SRAA_VALUE") == "Y") & 
            (F.coalesce(F.col("derived_location_value"), F.lit("")).isin(["CHN", ""])),
            "1"
        ).otherwise("0").alias("SRAT_VALUE"),
        F.current_timestamp().alias("SRAT_CREATION_DT"),
        F.lit("ELC").alias("SRAT_CREATIONUID"),
        F.current_timestamp().alias("SRAT_UPDATE_DT"),
        F.lit("ELC").alias("SRAT_UPDATEUID"),
        F.col("BATCH_ID"),
        F.col("task_id")
    )
    
    # 合并两个属性
    scflag_auxiliary_attributes_df = attr1_df.unionByName(attr2_df)

    if not scflag_auxiliary_attributes_df.isEmpty():
        append_table(scflag_auxiliary_attributes_df,f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_custom_attributes")

In [0]:
def parse_consumer_tables(task_id, task_batchlist_df):

    bronze_df = spark.read.format("delta").load(get_env_config("bronze_path_consumer"))

    # 取当前task_id下的Batch_id_list，然后取这些Batch_id_list的consumer数据
    row = task_batchlist_df.filter(F.col("task_id") == task_id).select("Batch_id_list").head()
    batch_numbers = row[0].split(",") if row else []
    print(f"Batch_id_list：{batch_numbers}")
    latest_batch_df = bronze_df.filter(F.col("batch_number").isin(batch_numbers))

    # 更新当前任务bronze数据的task_id (用于处理重试数据)
    latest_batch_df = latest_batch_df.withColumn("task_id", F.lit(task_id)).cache()

    calc_consumer_attributes_df(task_id, latest_batch_df)
    calc_skin_concerns_df(task_id, latest_batch_df)
    calc_makeup_concerns_df(task_id, latest_batch_df)
    calc_hairtype_df(task_id, latest_batch_df)
    calc_hair_concerns_df(task_id, latest_batch_df)
    calc_hobby_df(task_id, latest_batch_df)
    calc_email_df(task_id, latest_batch_df)
    calc_phone_df(task_id, latest_batch_df)
    calc_address_df(task_id, latest_batch_df)
    calc_optin_df(task_id, latest_batch_df)
    calc_cross_brand_df(task_id, latest_batch_df)
    calc_aux_attr_df(task_id, latest_batch_df)
    calc_tnc_df(task_id, latest_batch_df)
    calc_remark_df(task_id, latest_batch_df)
    calc_customer_group_df(task_id, latest_batch_df)
    calc_note_df(task_id, latest_batch_df)
    calc_program_df(task_id, latest_batch_df)
    # email、phone、address执行完后才能执行consumer
    calc_consumer_df(task_id, latest_batch_df)

    # 针对HKG市场，计算额外的属性数据
    calc_derivedlocation_custom_attributes_df(task_id, latest_batch_df)
    calc_scflag_nonchina_custom_attributes_df(task_id)
    calc_scflag_china_custom_attributes_df(task_id)
    calc_scflag_auxiliary_attributes_df(task_id)
    
    latest_batch_df.unpersist()

In [0]:
# 只计算最新批次数据
task_id = dbutils.widgets.get("task_id")
print(f"task_id: {task_id}")

t_task_batchlist_log = f"{get_env_config('config_database')}.t_task_batchlist_log"
print(t_task_batchlist_log)
task_batchlist_df = spark.table(t_task_batchlist_log)

with StepLogger("parse_consumer_tables", "02", "consumerlist", task_id=task_id) as logger:
    parse_consumer_tables(task_id, task_batchlist_df)